# SplineNAM: Additive cubic spline network

SplineNAM replaces feature MLPs with trainable cubic spline layers. Scalar preprocessing and optional identifiability centering keep each learned shape interpretable.


## Model


$$
f_j(x_j)=\sum_{k=1}^{K}\theta_{jk}B_{jk}(x_j),
\qquad
\eta(x)=\beta_0+\sum_jf_j(x_j),
$$

with optional roughness control proportional to squared adjacent coefficient differences.


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_preprocessing` and `categorical_preprocessing` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import SplineNAMRegressor


model = SplineNAMRegressor(
    n_knots=10,
    learn_knots=False,
    identify=True,
    smoothing=1e-3,
    interactions=(("x1", "x2"),),
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, **metrics})
    display(model.term_importance(X_test).head())


## Model-specific controls

`n_knots` controls basis resolution, `learn_knots` makes locations trainable, and `identify` centers shapes. SplineNAM requires scalar transformed features.


In [ ]:
if RUN_TRAINING:
    display(model.term_importance(X_test))
    display(model.model_complexity())
    model.plot_terms(X_test, pages=1)


## Task variants and limits

The current public surface is `SplineNAMRegressor`; classification and LSS are deliberately unsupported.
